# 03 · A Real API as a Tool

Last time the weather was fake. Now the tool calls a **real, free weather API** ([Open-Meteo](https://open-meteo.com), no API key needed).

Everything about tool calling stays the same. Only the function body changes. That is the point: **a tool can be anything Python can do**: call an API, read a file, query a database.

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()   # read your keys from the .env file
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to your .env file (see .env.example)"

client = OpenAI()        # finds OPENAI_API_KEY on its own
MODEL = "gpt-4o-mini"    # a small, cheap model is plenty for this module
print("✅ Ready")

✅ Ready


## The real function
It makes two web requests: one to find the city's location, and one to get the weather there.
Always test a tool **on its own** before giving it to a model.

In [2]:
import requests

def get_weather(city):
    """Get the current weather for a city using the free Open-Meteo API (no API key needed)."""

    # Step 1: turn the city name into latitude and longitude
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1},
        timeout=10,
    ).json()
    if "results" not in geo:
        return f"Sorry, I could not find a city called {city}."
    place = geo["results"][0]

    # Step 2: ask for the weather at that location
    data = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": place["latitude"],
            "longitude": place["longitude"],
            "current": "temperature_2m,precipitation",
        },
        timeout=10,
    ).json()
    now = data["current"]

    return f"{place['name']}: {now['temperature_2m']}°C, rain right now: {now['precipitation']} mm"

In [3]:
print(get_weather("Pune"))
print(get_weather("Zzzzxx"))   # a city that does not exist, to see the error message

Pune: 18.6°C, rain right now: 0.0 mm


Sorry, I could not find a city called Zzzzxx.


## Describe it and ask
The description is the same as before.

In [4]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather (temperature and rain) for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name, for example Pune"},
                },
                "required": ["city"],
            },
        },
    },
]

## The full round trip in one cell
These are the same steps as the last notebook: ask, read the request, run the tool, send the result back.

In [5]:
messages = [{"role": "user", "content": "Do I need an umbrella in Pune right now?"}]

# 1. Ask the model. It may request a tool.
response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
message = response.choices[0].message
tool_call = message.tool_calls[0]

# 2. Read the request and run our function
args = json.loads(tool_call.function.arguments)
print("Model asked for:", tool_call.function.name, args)
result = get_weather(**args)
print("Tool returned  :", result)

# 3. Send the result back and get the final answer
messages.append(message)
messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})
final = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)

print("\nFinal answer   :", final.choices[0].message.content)

Model asked for: get_weather {'city': 'Pune'}


Tool returned  : Pune: 18.6°C, rain right now: 0.0 mm



Final answer   : You do not need an umbrella in Pune right now, as there is no rain currently.


## Notice what the model did
You asked about an *umbrella*. The model worked out that it needed the weather, picked the right tool, filled in the city, and then **used the real rain figure to answer**.

> ⚠️ The model fills in the tool inputs itself, so always treat them as untrusted. Here a wrong city only gives an error message. If a tool could delete files or spend money, you would need to check the inputs first.

Next: give the model several tools and let it choose.